# Llama-3.1-8B-Instruct
## ~5000 samples stratified

In [1]:
%%capture
%pip install torch bitsandbytes transformers accelerate peft trl fsspec==2024.10.0 pyarrow==14.0.2 wandb
%pip freeze > requirements.txt

In [2]:
# Parameters
n_sample = 200
n_epochs = 1

In [3]:
import numpy as np
import pandas as pd
import os
from tqdm import tqdm
import bitsandbytes as bnb
import torch
import torch.nn as nn
import transformers
from datasets import Dataset
from peft import LoraConfig, PeftConfig
from trl import SFTTrainer
from trl import setup_chat_format
from transformers import (AutoModelForCausalLM, 
                          AutoTokenizer, 
                          BitsAndBytesConfig, 
                          TrainingArguments, 
                          pipeline, 
                          logging)
from sklearn.metrics import (accuracy_score,
                             classification_report,
                             confusion_matrix,
                             precision_recall_fscore_support)
from sklearn.model_selection import train_test_split

from data import go_emotions
from credentials import hf_token, wandb_key

## Load data

In [4]:
with open('data/go_emotions/emotions.txt', "r") as file:
    lines = file.readlines()

# Remove any trailing newline characters
l_emotions = [line.strip() for line in lines]

d_go_emotions = {i:e for i,e in enumerate(l_emotions)}

In [5]:
df_train = go_emotions['train']
df_test = go_emotions['test']
df_val = go_emotions['val']

In [6]:
for data in [df_train, df_test, df_val]:
    data['l_emotions'] = data['labels'].apply(lambda x: [d_go_emotions[label] for label in x])
    data['emotions'] = data['l_emotions'].apply(str)

In [7]:
# Define the prompt generation functions
def generate_prompt(data_point):
    return f"""
Classify the text into {', '.join(l_emotions)}
Return the answer as the corresponding emotion label.
text: {data_point["text"]}
label: {data_point["emotions"]}""".strip()

def generate_test_prompt(data_point):
    return f"""
Classify the text into {', '.join(l_emotions)}
Return the answer as the corresponding emotion label.
text: {data_point["text"]}
label: """.strip()

# Generate prompts for training and evaluation data
df_train.loc[:,'prompt'] = df_train.apply(generate_prompt, axis=1)
df_val.loc[:,'prompt'] = df_val.apply(generate_prompt, axis=1)

# Generate test prompts and extract true labels
y_test = df_test.loc[:,'emotions']
X_test = pd.DataFrame(df_test.apply(generate_test_prompt, axis=1), columns=["prompt"])

In [8]:
df_train.emotions.value_counts()

emotions
['neutral']                                  12823
['admiration']                                2710
['approval']                                  1873
['gratitude']                                 1857
['amusement']                                 1652
                                             ...  
['disgust', 'excitement']                        1
['admiration', 'caring', 'optimism']             1
['disappointment', 'optimism', 'sadness']        1
['excitement', 'joy', 'pride']                   1
['desire', 'disgust', 'love']                    1
Name: count, Length: 711, dtype: int64

In [9]:
# Stratified sample
sample_idx = []
stratified_sample = df_train.sample(frac=1, random_state=42).reset_index(drop=True)
shuffle = stratified_sample
for emo in l_emotions:
    idx = list(shuffle[shuffle['emotions'].str.contains(emo)].index[0:n_sample])
    sample_idx += idx
    shuffle = shuffle.drop(idx)

stratified_sample = stratified_sample.iloc[sample_idx]

print(len(stratified_sample))
stratified_sample.emotions.value_counts()

5269


emotions
['disapproval']                            215
['neutral']                                200
['sadness']                                175
['love']                                   172
['gratitude']                              162
                                          ... 
['amusement', 'confusion', 'surprise']       1
['amusement', 'anger']                       1
['amusement', 'excitement', 'surprise']      1
['amusement', 'joy', 'neutral']              1
['amusement', 'annoyance']                   1
Name: count, Length: 442, dtype: int64

In [10]:
# Convert to datasets
train_data = Dataset.from_pandas(stratified_sample[["prompt"]])
eval_data = Dataset.from_pandas(df_val[["prompt"]])

In [11]:
train_data['prompt'][3]

"Classify the text into admiration, amusement, anger, annoyance, approval, caring, confusion, curiosity, desire, disappointment, disapproval, disgust, embarrassment, excitement, fear, gratitude, grief, joy, love, nervousness, optimism, pride, realization, relief, remorse, sadness, surprise, neutral\nReturn the answer as the corresponding emotion label.\ntext: He’s one of the calmest goalies I’ve ever seen in general haha\nlabel: ['admiration', 'amusement']"

In [12]:
base_model_name = "meta-llama/Llama-3.1-8B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=False,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype="float16",
)

model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    device_map="auto",
    torch_dtype="float16",
    quantization_config=bnb_config, 
    token=hf_token
)

model.config.use_cache = False
model.config.pretraining_tp = 1

tokenizer = AutoTokenizer.from_pretrained(
    base_model_name, 
    token=hf_token
)

tokenizer.pad_token_id = tokenizer.eos_token_id

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [13]:
def predict(test, model, tokenizer):
    y_pred = []
    categories = l_emotions

    transformers.logging.set_verbosity_error()
    
    for i in tqdm(range(len(test)), leave=True):
        prompt = test.iloc[i]["prompt"]
        pipe = pipeline(
                    task="text-generation",
                    model=model,
                    tokenizer=tokenizer,
                    max_new_tokens=5,
                    temperature=0.1
                )

        result = pipe(prompt)
        answer = result[0]['generated_text'].split("label:")[-1].strip()

        y_labels = []

        # Determine the predicted category
        for category in categories:
            if category.lower() in answer.replace(r'\n', ' ').lower():
                y_labels.append(category)

        y_pred.append(y_labels)

    return y_pred

In [14]:
# Model evaluation function
def model_eval(y_true, y_pred_labels, emotions):
    # Create matrix labels
    def create_matrix_labels(labels, emotions=l_emotions):
        matrix_labels = np.zeros((len(labels), len(emotions)))
        for i, label in enumerate(labels):
            for j, emo in enumerate(emotions):
                if emo in label:
                    matrix_labels[i, j] = 1
        return matrix_labels

    # Reshape
    y_true = create_matrix_labels(y_true)
    y_pred_labels = create_matrix_labels(y_pred_labels)

    # Defining variables
    precision = []
    recall = []
    f1 = []

    # Per emotion evaluation
    idx2emotion = {i: e for i, e in enumerate(emotions)}

    for i in range(len(emotions)):

        # Computing precision, recall and f1-score
        p, r, f1_score, _ = precision_recall_fscore_support(y_true[:, i], y_pred_labels[:, i], average="binary")

        # Append results in lists
        precision.append(round(p, 2))
        recall.append(round(r, 2))
        f1.append(round(f1_score, 2))

    # Macro evaluation
    macro_p, macro_r, macro_f1_score, _ = precision_recall_fscore_support(y_true, y_pred_labels, average="macro")

    # Append results in lists
    precision.append(round(macro_p, 2))
    recall.append(round(macro_r, 2))
    f1.append(round(macro_f1_score, 2))

    # Converting results to a dataframe
    df_results = pd.DataFrame({"Precision":precision, "Recall":recall, 'F1':f1})
    df_results.index = emotions+['MACRO-AVERAGE']

    return df_results

In [ ]:
import wandb

wandb.login(key=wandb_key)
run = wandb.init(
    project='Fine-tune llama-3.1-8B-Instruct on Go Emotions',
    job_type="training",
    anonymous="allow"
)

In [16]:
def find_all_linear_names(model):
    cls = bnb.nn.Linear4bit
    lora_module_names = set()
    for name, module in model.named_modules():
        if isinstance(module, cls):
            names = name.split('.')
            lora_module_names.add(names[0] if len(names) == 1 else names[-1])
    if 'lm_head' in lora_module_names:  # needed for 16 bit
        lora_module_names.remove('lm_head')
    return list(lora_module_names)
modules = find_all_linear_names(model)
modules

['v_proj', 'gate_proj', 'down_proj', 'q_proj', 'up_proj', 'k_proj', 'o_proj']

In [17]:
output_dir="llama-3.1-fine-tuned-model"

peft_config = LoraConfig(
    lora_alpha=16,
    lora_dropout=0,
    r=64,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=modules,
)

training_arguments = TrainingArguments(
    output_dir=output_dir,                    # directory to save and repository id
    num_train_epochs=n_epochs,                # number of training epochs
    per_device_train_batch_size=1,            # batch size per device during training
    gradient_accumulation_steps=8,            # number of steps before performing a backward/update pass
    gradient_checkpointing=True,              # use gradient checkpointing to save memory
    optim="paged_adamw_32bit",
    logging_steps=1,
    learning_rate=2e-4,                       # learning rate, based on QLoRA paper
    weight_decay=0.001,
    fp16=True,
    bf16=False,
    max_grad_norm=0.3,                        # max gradient norm based on QLoRA paper
    max_steps=-1,
    warmup_ratio=0.03,                        # warmup ratio based on QLoRA paper
    group_by_length=False,
    lr_scheduler_type="cosine",               # use cosine learning rate scheduler
    report_to="wandb",                  # report metrics to w&b
    eval_strategy="steps",              # save checkpoint every epoch
    eval_steps = 0.2
)

trainer = SFTTrainer(
    model=model,
    args=training_arguments,
    train_dataset=train_data,
    eval_dataset=eval_data,
    peft_config=peft_config,
    dataset_text_field="prompt",
    tokenizer=tokenizer,
    max_seq_length=512,
    packing=False,
    dataset_kwargs={
    "add_special_tokens": False,
    "append_concat_token": False,
    }
)

/opt/conda/lib/python3.10/site-packages/huggingface_hub/utils/_deprecation.py:100: FutureWarning: Deprecated argument(s) used in '__init__': dataset_text_field, max_seq_length, dataset_kwargs. Will not be supported from version '1.0.0'.

Deprecated positional argument(s) used in SFTTrainer, please use the SFTConfig to set these arguments instead.
  warnings.warn(message, FutureWarning)
/opt/conda/lib/python3.10/site-packages/trl/trainer/sft_trainer.py:283: UserWarning: You passed a `max_seq_length` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(
/opt/conda/lib/python3.10/site-packages/trl/trainer/sft_trainer.py:321: UserWarning: You passed a `dataset_text_field` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(
/opt/conda/lib/python3.10/site-packages/trl/trainer/sft_trainer.py:327: UserWarning: You passed a `dataset_kwargs` argument to the SFTTrainer, the value you pass

Map:   0%|          | 0/5269 [00:00<?, ? examples/s]

Map:   0%|          | 0/5426 [00:00<?, ? examples/s]

/opt/conda/lib/python3.10/site-packages/trl/trainer/sft_trainer.py:401: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `SFTTrainer.__init__`. Use `processing_class` instead.
  super().__init__(


In [18]:
trainer.train()

wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
/opt/conda/lib/python3.10/site-packages/torch/_dynamo/eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss,Validation Loss
132,0.515000,0.664682
264,0.553600,0.661231
396,0.630500,0.655066
528,0.625500,0.653602


/opt/conda/lib/python3.10/site-packages/peft/utils/other.py:716: UserWarning: Unable to fetch remote file due to the following error 401 Client Error. (Request ID: Root=1-677a3b9b-14bd92734fcd738571a5ee74;8967fc56-46c9-40e4-bd63-0b30e9119463)

Cannot access gated repo for url https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct/resolve/main/config.json.
Access to model meta-llama/Llama-3.1-8B-Instruct is restricted. You must have access to it and be authenticated to access it. Please log in. - silently ignoring the lookup for the file config.json in meta-llama/Llama-3.1-8B-Instruct.
  warnings.warn(
/opt/conda/lib/python3.10/site-packages/peft/utils/save_and_load.py:246: UserWarning: Could not find a config file in meta-llama/Llama-3.1-8B-Instruct - will assume that the vocabulary was not modified.
  warnings.warn(
/opt/conda/lib/python3.10/site-packages/torch/_dynamo/eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In 

TrainOutput(global_step=658, training_loss=0.6796824421654356, metrics={'train_runtime': 3282.0024, 'train_samples_per_second': 1.605, 'train_steps_per_second': 0.2, 'total_flos': 2.385314078301389e+16, 'train_loss': 0.6796824421654356, 'epoch': 0.9990510533308028})

In [19]:
wandb.finish()
model.config.use_cache = True

wandb:                                                                                
wandb: 
wandb: Run history:
wandb:               eval/loss █▆▂▁
wandb:            eval/runtime █▇▁▇
wandb: eval/samples_per_second ▁▃█▃
wandb:   eval/steps_per_second ▁▁█▁
wandb:             train/epoch ▁▁▁▁▁▂▂▂▂▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▅▆▆▆▆▆▆▆▇▇█████
wandb:       train/global_step ▁▁▁▁▁▂▂▂▂▂▃▃▃▃▃▃▃▃▄▄▄▄▅▅▅▆▆▆▆▆▇▇▇▇▇▇▇▇██
wandb:         train/grad_norm █▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
wandb:     train/learning_rate ▄████▇▇▆▆▆▆▆▆▆▅▅▅▅▅▅▄▄▃▃▃▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁
wandb:              train/loss █▃▅▃▅▅▄▆▆▂▄▅▂▄▃▃▂▁▁▃▃▃▂▄▄▃▄▃▂▃▄▂▄▄▄▂▅▃▃▅
wandb: 
wandb: Run summary:
wandb:                eval/loss 0.6536
wandb:             eval/runtime 161.1775
wandb:  eval/samples_per_second 33.665
wandb:    eval/steps_per_second 4.213
wandb:               total_flos 2.385314078301389e+16
wandb:              train/epoch 0.99905
wandb:        train/global_step 658
wandb:          train/grad_norm 0.1316
wandb:      train/learning_

In [20]:
y_pred = predict(X_test, model, tokenizer)
model_eval(y_test, y_pred, l_emotions)

  0%|          | 0/5427 [00:00<?, ?it/s]/opt/conda/lib/python3.10/site-packages/torch/_dynamo/eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/opt/conda/lib/python3.10/site-packages/torch/utils/checkpoint.py:87: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(
100%|██████████| 5427/5427 [1:20:58<00:00,  1.12it/s]


,Precision,Recall,F1
admiration,0.66,0.61,0.64
amusement,0.76,0.71,0.73
anger,0.40,0.45,0.43
annoyance,0.55,0.14,0.23
approval,0.33,0.51,0.40
caring,0.43,0.41,0.42
confusion,0.40,0.42,0.41
curiosity,0.42,0.60,0.49
desire,0.40,0.52,0.45
disappointment,0.18,0.30,0.23
